In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import os
import random
import datetime
import re

# Replace with your API key
API_KEY = "di isi dengan API key kamu"

def extract_folder_id(drive_url):
    # Regular expression to extract the folder ID from the URL
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', drive_url)
    if match:
        return match.group(1)
    else:
        raise ValueError("Invalid Google Drive folder URL")


def generate_random_duration():
    # Durasi dalam detik, antara 30 detik dan 5 menit (300 detik)
    duration_seconds = random.randint(30, 300)
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes:02}:{seconds:02}"


def escape_sql_string(value):
    # Escape single quotes by doubling them
    return value.replace("'", "''")


def get_artist_from_title(title, list_title_artist):
    """Fungsi untuk mencocokkan judul dan menemukan artis yang sesuai."""
    for item in list_title_artist:
        # Pisahkan berdasarkan ' --- '
        song_title, artist = item.split(' --- ')
        if title.lower() == song_title.lower():  # Cocokkan tanpa memperhatikan huruf besar/kecil
            return artist
    return ""  # Return empty string jika tidak ditemukan


def list_files_in_folder(api_key, drive_url):
    try:
        folder_id = extract_folder_id(drive_url)
        service = build("drive", "v3", developerKey=api_key)

        query = f"'{folder_id}' in parents"
        response = service.files().list(q=query, fields="files(id, name)").execute()
        files = response.get("files", [])

        if not files:
            print("No files found.")
            return

        # Starting timestamp for date_added
        base_time = datetime.datetime.now().replace(microsecond=0)

        # Base SQL INSERT statement template
        insert_statement_template = "INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES \n"

        # Daftar pasangan nama lagu dan artis
        list_title_artist = [
        ]

        # Parameters for the static fields
        # 1 = IndoPride
        # 2 = 日本の歌
        # 3 = Javasong
        # 4 = Worldwide
        category = "2"
        # hati-hati dengan karakter khusus dalam string SQL, seperti tanda kutip tunggal
        # absolutely tidak boleh ada bentuk kutip, baik tunggal/ganda atau backtick sekali pun
        # tapi kenapa di album bisa, ya?

        is_dynamic_artist = False
        # is_dynamic_artist = True
        artist = "Sukima Switch"
        album = 'Sukima Switch - LINE'
        cover_link = "https://drive.google.com/file/d/1HNZ7TKPfSDjz5Jdq3nGZ8llxTSeUIEBK/view?usp=drive_link"
        favorite = "0"

        # Collecting each row of values
        values = []

        for index, file in enumerate(files):
            file_name = file.get("name")
            file_name_without_extension, _ = os.path.splitext(file_name)
            file_id = file.get("id")
            link_gdrive = (
                f"https://drive.google.com/file/d/{file_id}/view?usp=drive_link"
            )
            date_added = base_time + datetime.timedelta(seconds=index)
            date_added_str = date_added.strftime("%Y-%m-%d %H:%M:%S")

            # Jika artist kosong, cari artist berdasarkan nama file
            if is_dynamic_artist:
                artist = get_artist_from_title(
                    file_name_without_extension, list_title_artist)

            duration = generate_random_duration()

            # Escape special characters in the values
            file_name_without_extension = escape_sql_string(
                file_name_without_extension)
            link_gdrive = escape_sql_string(link_gdrive)
            artist = escape_sql_string(artist)
            album = escape_sql_string(album)
            cover_link = escape_sql_string(cover_link)
            date_added_str = escape_sql_string(date_added_str)

            values.append(
                f"(NULL, '{category}', '{link_gdrive}', '{file_name_without_extension}', '{artist}', '{album}', '{duration}', '{cover_link}', '{favorite}', '{date_added_str}')"
            )

        # Joining all values into the final SQL INSERT statement
        insert_statement = insert_statement_template + ",\n".join(values) + ";"
        print(insert_statement)

    except HttpError as error:
        print(f"An error occurred: {error}")
    except ValueError as error:
        print(f"An error occurred: {error}")


if __name__ == "__main__":
    # Replace with your full Google Drive folder URL
    FOLDER_URL = "https://drive.google.com/drive/folders/15T_Tx3MAPcdc5GsEUYczh4VG2cIeIUj4?usp=drive_link"
    list_files_in_folder(API_KEY, FOLDER_URL)

INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES 
(NULL, '2', 'https://drive.google.com/file/d/1BY2RG1i0iMC_huSd1UB39yAyXGrKYKO9/view?usp=drive_link', 'LINE', 'Sukima Switch', 'Sukima Switch - LINE', '03:18', 'https://drive.google.com/file/d/1HNZ7TKPfSDjz5Jdq3nGZ8llxTSeUIEBK/view?usp=drive_link', '0', '2025-07-15 16:37:21'),
(NULL, '2', 'https://drive.google.com/file/d/10FTEbYH6i3rSezHJNBktn_aQ9BErtQZJ/view?usp=drive_link', 'LINE (Backing Track)', 'Sukima Switch', 'Sukima Switch - LINE', '01:29', 'https://drive.google.com/file/d/1HNZ7TKPfSDjz5Jdq3nGZ8llxTSeUIEBK/view?usp=drive_link', '0', '2025-07-15 16:37:22'),
(NULL, '2', 'https://drive.google.com/file/d/1PjIqykCDYgwkonMHeqUJjeGpO-BaUv5F/view?usp=drive_link', 'Hanatsu (Backing Track)', 'Sukima Switch', 'Sukima Switch - LINE', '02:49', 'https://drive.google.com/file/d/1HNZ7TKPfSDjz5Jdq3nGZ8llxTSeUIEBK/view?usp=drive_link', '0', '2025-07-15 16:37:23'),
(NULL, '2', 'http